# 🚀 Модуль 2. Урок 3. Глубокое погружение в JOIN: исключение дубликатов и декартова произведения

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_02_sql/lesson_03_joins_deep_dive.ipynb)

## 📌 Бизнес-контекст и методология
Объединение таблиц (`INNER`, `LEFT`, `FULL`) лежит в основе аналитических запросов.

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

db_path = ensure_dataset("ecommerce.db")
conn = sqlite3.connect(db_path)
print(f"🟢 Подключение к базе данных успешно установлено: {db_path}")


🟢 Подключение к базе данных успешно установлено: course\datasets\ecommerce.db


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(ensure_dataset("ecommerce.db"))

query = """
SELECT 
    p.category,
    COUNT(DISTINCT o.order_id) AS orders_count,
    SUM(oi.quantity) AS units_sold,
    ROUND(SUM(oi.total_amount), 2) AS category_revenue,
    ROUND(SUM(oi.quantity * (p.price - p.cost)), 2) AS category_profit
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
JOIN orders o ON oi.order_id = o.order_id
WHERE o.status = 'Completed'
GROUP BY p.category
ORDER BY category_revenue DESC;
"""

df_cats = pd.read_sql_query(query, conn)
df_cats


,category,orders_count,units_sold,category_revenue,category_profit
0,Electronics,876,1260,740144.93,412526.82
1,Fashion,727,1027,719947.13,332465.24
2,Beauty,608,823,602967.07,237244.80
3,Home & Kitchen,694,970,547300.48,272545.93


### 🎯 Практическое задание 1: Поиск клиентов без заказов
Используя `LEFT JOIN`, найдите клиентов без совершенных заказов.

In [3]:
### ВАШ КОД ЗДЕСЬ
query_no_orders = """
SELECT 
    c.acquisition_channel,
    COUNT(c.customer_id) AS inactive_customers
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL
GROUP BY c.acquisition_channel
ORDER BY inactive_customers DESC;
"""

df_inactive = pd.read_sql_query(query_no_orders, conn)
print(df_inactive)

assert not df_inactive.empty, "Датафрейм не должен быть пустым"
assert 'inactive_customers' in df_inactive.columns, "Колонка обязательна"
print("✅ Тест успешно пройден!")


  acquisition_channel  inactive_customers
0            Paid Ads                  56
1              Direct                  44
2        Social Media                  41
3    Email Newsletter                  33
4            Referral                  32
5      Organic Search                  26
✅ Тест успешно пройден!


### 💡 Памятка по типам JOIN
LEFT JOIN сохраняет все строки левой таблицы, заменяя несовпавшие значения на NULL.